# פרויקט גמר - מערכת סוכנים למערכת תשלומים דמוית Bit

המחברת מממשת סימולציה מלאה של מערכת תשלומים ללא חיבור לשירותי בנק, כרטיסי אשראי או תשלומים אמיתיים. המערכת כוללת RouterAgent, OrchestratorAgent, בחירת כלים, זיכרון קצר טווח עסקי, מערכת תשלומים, בדיקות הונאה, ביקורת אבטחה, הסברים, ביקורת איכות ומנגנון גיבוי.

נבחרו שני אלמנטים מתקדמים: PolicyAgent להגבלת סכומים, ושמירה/טעינה מ-JSON.

## מימוש המערכת והסוכנים

כל סוכן מחזיר תוצאה במבנה אחיד מסוג AgentResult. מערכת התשלומים עצמה מופרדת מהסוכנים כדי לאפשר בדיקות ברורות וניהול זרימה על ידי OrchestratorAgent.

In [ ]:
from dataclasses import dataclass, field, asdict
from typing import Any, Optional, Dict, List
from datetime import datetime, date
import json
import re


@dataclass
class AgentResult:
    agent_name: str
    output: Any
    confidence: float = 1.0
    metadata: Optional[Dict[str, Any]] = None


@dataclass
class User:
    user_id: str
    name: str
    phone_number: str


@dataclass
class Wallet:
    user_id: str
    balance: float


@dataclass
class Transaction:
    transaction_id: str
    sender_id: str
    receiver_id: str
    amount: float
    timestamp: str
    status: str
    risk_score: float = 0.0
    reason: str = ""


@dataclass
class PaymentRequest:
    request_id: str
    requester_id: str
    payer_id: str
    amount: float
    status: str
    created_at: str


class ShortTermMemory:
    def __init__(self, limit: int = 10):
        self.limit = limit
        self.last_action = None
        self.last_user = None
        self.last_transaction = None
        self.last_payment_request = None
        self.last_result = None
        self.recent_actions = []

    def update(self, action: str, result: AgentResult, user_id=None, transaction=None, payment_request=None):
        self.last_action = action
        self.last_result = result
        if user_id:
            self.last_user = user_id
        if transaction:
            self.last_transaction = transaction
        if payment_request:
            self.last_payment_request = payment_request
        self.recent_actions.append({
            "time": datetime.now().isoformat(timespec="seconds"),
            "action": action,
            "result": result.output,
        })
        self.recent_actions = self.recent_actions[-self.limit:]


class PaymentSystem:
    def __init__(self):
        self.users: Dict[str, User] = {}
        self.wallets: Dict[str, Wallet] = {}
        self.transactions: Dict[str, Transaction] = {}
        self.payment_requests: Dict[str, PaymentRequest] = {}
        self.audit_log: List[Dict[str, Any]] = []
        self._user_counter = 1
        self._transaction_counter = 1
        self._request_counter = 1

    def _now(self):
        return datetime.now().isoformat(timespec="seconds")

    def _audit(self, action, details):
        self.audit_log.append({"time": self._now(), "action": action, "details": details})

    def create_user(self, name, phone_number, initial_balance=0):
        if initial_balance < 0:
            self._audit("create_user_failed", {"name": name, "reason": "negative initial balance"})
            raise ValueError("Cannot create a user with negative initial balance")
        user_id = f"U{self._user_counter:03d}"
        self._user_counter += 1
        user = User(user_id, name, phone_number)
        self.users[user_id] = user
        self.wallets[user_id] = Wallet(user_id, float(initial_balance))
        self._audit("create_user", asdict(user) | {"initial_balance": float(initial_balance)})
        return user

    def get_balance(self, user_id):
        if user_id not in self.wallets:
            raise ValueError("User does not exist")
        self._audit("get_balance", {"user_id": user_id, "balance": self.wallets[user_id].balance})
        return self.wallets[user_id].balance

    def transfer_money(self, sender_id, receiver_id, amount):
        amount = float(amount)
        if amount <= 0:
            self._audit("transfer_failed", {"sender_id": sender_id, "receiver_id": receiver_id, "amount": amount, "reason": "non-positive amount"})
            raise ValueError("Transfer amount must be positive")
        if sender_id not in self.users or receiver_id not in self.users:
            self._audit("transfer_failed", {"sender_id": sender_id, "receiver_id": receiver_id, "amount": amount, "reason": "missing user"})
            raise ValueError("Both sender and receiver must exist")
        if sender_id == receiver_id:
            self._audit("transfer_failed", {"sender_id": sender_id, "amount": amount, "reason": "self transfer"})
            raise ValueError("Cannot transfer money to yourself")
        if self.wallets[sender_id].balance < amount:
            self._audit("transfer_failed", {"sender_id": sender_id, "amount": amount, "reason": "insufficient balance"})
            raise ValueError("Insufficient balance")

        self.wallets[sender_id].balance -= amount
        self.wallets[receiver_id].balance += amount
        transaction_id = f"T{self._transaction_counter:04d}"
        self._transaction_counter += 1
        transaction = Transaction(transaction_id, sender_id, receiver_id, amount, self._now(), "approved")
        self.transactions[transaction_id] = transaction
        self._audit("transfer_money", asdict(transaction))
        return transaction

    def get_transactions(self, user_id):
        if user_id not in self.users:
            raise ValueError("User does not exist")
        result = [tx for tx in self.transactions.values() if tx.sender_id == user_id or tx.receiver_id == user_id]
        self._audit("get_transactions", {"user_id": user_id, "count": len(result)})
        return result

    def request_payment(self, requester_id, payer_id, amount):
        amount = float(amount)
        if amount <= 0:
            self._audit("request_payment_failed", {"requester_id": requester_id, "payer_id": payer_id, "amount": amount, "reason": "non-positive amount"})
            raise ValueError("Payment request amount must be positive")
        if requester_id not in self.users or payer_id not in self.users:
            self._audit("request_payment_failed", {"requester_id": requester_id, "payer_id": payer_id, "reason": "missing user"})
            raise ValueError("Both requester and payer must exist")
        request_id = f"R{self._request_counter:04d}"
        self._request_counter += 1
        request = PaymentRequest(request_id, requester_id, payer_id, amount, "pending", self._now())
        self.payment_requests[request_id] = request
        self._audit("request_payment", asdict(request))
        return request

    def approve_payment_request(self, request_id):
        if request_id not in self.payment_requests:
            raise ValueError("Payment request does not exist")
        request = self.payment_requests[request_id]
        if request.status != "pending":
            self._audit("approve_request_failed", {"request_id": request_id, "status": request.status})
            raise ValueError("Cannot approve a request that was already approved or rejected")
        transaction = self.transfer_money(request.payer_id, request.requester_id, request.amount)
        request.status = "approved"
        self._audit("approve_payment_request", {"request_id": request_id, "transaction_id": transaction.transaction_id})
        return request, transaction

    def reject_payment_request(self, request_id):
        if request_id not in self.payment_requests:
            raise ValueError("Payment request does not exist")
        request = self.payment_requests[request_id]
        if request.status != "pending":
            self._audit("reject_request_failed", {"request_id": request_id, "status": request.status})
            raise ValueError("Cannot reject a request that was already approved or rejected")
        request.status = "rejected"
        self._audit("reject_payment_request", asdict(request))
        return request

    def save_to_json(self, path):
        data = {
            "users": {k: asdict(v) for k, v in self.users.items()},
            "wallets": {k: asdict(v) for k, v in self.wallets.items()},
            "transactions": {k: asdict(v) for k, v in self.transactions.items()},
            "payment_requests": {k: asdict(v) for k, v in self.payment_requests.items()},
            "audit_log": self.audit_log,
            "counters": {"user": self._user_counter, "transaction": self._transaction_counter, "request": self._request_counter},
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)
        self._audit("save_to_json", {"path": path})

    def load_from_json(self, path):
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        self.users = {k: User(**v) for k, v in data["users"].items()}
        self.wallets = {k: Wallet(**v) for k, v in data["wallets"].items()}
        self.transactions = {k: Transaction(**v) for k, v in data["transactions"].items()}
        self.payment_requests = {k: PaymentRequest(**v) for k, v in data["payment_requests"].items()}
        self.audit_log = data["audit_log"]
        self._user_counter = data["counters"]["user"]
        self._transaction_counter = data["counters"]["transaction"]
        self._request_counter = data["counters"]["request"]
        self._audit("load_from_json", {"path": path})


class RouterAgent:
    def run(self, message: str) -> AgentResult:
        text = message.lower()
        rules = [
            ("createUser", ["create user", "new user", "add user", "משתמש"]),
            ("checkBalance", ["balance", "יתרה"]),
            ("transferMoney", ["transfer", "send money", "pay", "העבר"]),
            ("requestPayment", ["request payment", "payment request", "בקש"]),
            ("approvePayment", ["approve", "אשר"]),
            ("rejectPayment", ["reject", "דחה"]),
            ("showTransactions", ["transactions", "history", "היסטוריה"]),
            ("fraudCheck", ["fraud", "suspicious", "חשוד"]),
            ("securityReview", ["security", "audit", "אבטחה"]),
            ("explainLastAction", ["explain", "last", "הסבר"]),
        ]
        for intent, keywords in rules:
            if any(keyword in text for keyword in keywords):
                return AgentResult("RouterAgent", intent, 0.95)
        return AgentResult("RouterAgent", "unknown", 0.3)


class ToolSelector:
    def run(self, intent: str) -> AgentResult:
        mapping = {
            "createUser": "PaymentAgent",
            "checkBalance": "PaymentAgent",
            "transferMoney": "PaymentAgent",
            "requestPayment": "PaymentAgent",
            "approvePayment": "PaymentAgent",
            "rejectPayment": "PaymentAgent",
            "showTransactions": "PaymentAgent",
            "fraudCheck": "FraudDetectionAgent",
            "securityReview": "SecurityAgent",
            "explainLastAction": "ExplanationAgent",
            "unknown": "FallbackAgent",
        }
        return AgentResult("ToolSelector", mapping.get(intent, "FallbackAgent"), 0.9)


class PolicyAgent:
    def __init__(self, max_single_transfer=5000, daily_limit=10000):
        self.max_single_transfer = max_single_transfer
        self.daily_limit = daily_limit

    def run(self, system: PaymentSystem, sender_id: str, amount: float) -> AgentResult:
        amount = float(amount)
        today = date.today().isoformat()
        sent_today = sum(
            tx.amount for tx in system.transactions.values()
            if tx.sender_id == sender_id and tx.timestamp.startswith(today) and tx.status == "approved"
        )
        if amount > self.max_single_transfer:
            return AgentResult("PolicyAgent", "Rejected by policy: single transfer limit exceeded", 0.95, {"allowed": False})
        if sent_today + amount > self.daily_limit:
            return AgentResult("PolicyAgent", "Rejected by policy: daily transfer limit exceeded", 0.95, {"allowed": False})
        return AgentResult("PolicyAgent", "Policy approved", 0.95, {"allowed": True, "sent_today": sent_today})


class FraudDetectionAgent:
    def run(self, system: PaymentSystem, transaction: Optional[Transaction] = None, user_id: Optional[str] = None) -> AgentResult:
        if transaction is None:
            txs = system.get_transactions(user_id) if user_id else list(system.transactions.values())
            transaction = txs[-1] if txs else None
        if transaction is None:
            return AgentResult("FraudDetectionAgent", "No transaction to review", 0.8, {"suspicious": False})

        risk = 0.0
        reasons = []
        if transaction.amount >= 4000:
            risk += 0.4
            reasons.append("large amount")
        current_balance = system.wallets[transaction.sender_id].balance
        if transaction.amount > max(current_balance, 1):
            risk += 0.25
            reasons.append("high amount compared to remaining balance")
        sender_transactions = [tx for tx in system.transactions.values() if tx.sender_id == transaction.sender_id]
        if len(sender_transactions) >= 3:
            risk += 0.25
            reasons.append("many recent sender actions")
        risk = min(risk, 1.0)
        transaction.risk_score = risk
        transaction.reason = ", ".join(reasons) if reasons else "normal behavior"
        suspicious = risk >= 0.6
        system._audit("fraud_check", {"transaction_id": transaction.transaction_id, "risk_score": risk, "suspicious": suspicious})
        return AgentResult("FraudDetectionAgent", {"transaction_id": transaction.transaction_id, "risk_score": risk, "suspicious": suspicious, "reasons": reasons}, 0.9, {"suspicious": suspicious})


class SecurityAgent:
    def run(self, system: PaymentSystem) -> AgentResult:
        checks = {
            "users_have_wallets": set(system.users) == set(system.wallets),
            "audit_log_exists": isinstance(system.audit_log, list),
            "transactions_are_logged": any(row["action"] == "transfer_money" for row in system.audit_log) or len(system.transactions) == 0,
            "no_negative_wallets": all(wallet.balance >= 0 for wallet in system.wallets.values()),
            "requests_have_valid_status": all(req.status in {"pending", "approved", "rejected"} for req in system.payment_requests.values()),
        }
        passed = all(checks.values())
        return AgentResult("SecurityAgent", checks, 0.95 if passed else 0.55, {"passed": passed})


class ExplanationAgent:
    def run(self, memory: ShortTermMemory) -> AgentResult:
        tx = memory.last_transaction
        if tx:
            suspicious_text = " סומנה כחשודה" if tx.risk_score >= 0.6 else " לא סומנה כחשודה"
            explanation = (
                f"העסקה האחרונה היא {tx.transaction_id}: משתמש {tx.sender_id} העביר {tx.amount} למשתמש {tx.receiver_id}. "
                f"הסטטוס הוא {tx.status}, ציון הסיכון הוא {tx.risk_score:.2f}, ולכן העסקה{suspicious_text}. "
                f"נימוק: {tx.reason or 'לא נדרשה הערת סיכון'}."
            )
        elif memory.last_payment_request:
            req = memory.last_payment_request
            explanation = f"בקשת התשלום האחרונה היא {req.request_id}, סכום {req.amount}, סטטוס {req.status}."
        else:
            explanation = "אין עדיין פעולה אחרונה שניתן להסביר."
        return AgentResult("ExplanationAgent", explanation, 0.9)


class CriticAgent:
    def run(self, result: AgentResult) -> AgentResult:
        needs_fix = result.confidence < 0.6 or (isinstance(result.output, str) and "error" in result.output.lower())
        return AgentResult("CriticAgent", {"accepted": not needs_fix, "reviewed_agent": result.agent_name}, 0.85, {"needs_fix": needs_fix})


class FallbackAgent:
    def run(self, message: str) -> AgentResult:
        return AgentResult("FallbackAgent", "I could not understand the payment intent. Please specify user, amount, or action more clearly.", 0.7)


class ReflectionAgent:
    def run(self, error_message: str) -> AgentResult:
        suggestions = {
            "positive": "Use an amount greater than zero.",
            "Insufficient": "Try a smaller amount or add balance first.",
            "exist": "Check that both user IDs exist before running the action.",
            "yourself": "Choose a different receiver user ID.",
            "already": "Use a pending payment request ID.",
        }
        suggestion = "Check the input values and try again."
        for key, value in suggestions.items():
            if key.lower() in error_message.lower():
                suggestion = value
                break
        return AgentResult("ReflectionAgent", {"error": error_message, "suggestion": suggestion}, 0.85)


class PaymentAgent:
    def run(self, system: PaymentSystem, intent: str, **kwargs) -> AgentResult:
        try:
            if intent == "createUser":
                user = system.create_user(kwargs["name"], kwargs["phone_number"], kwargs.get("initial_balance", 0))
                return AgentResult("PaymentAgent", asdict(user), 0.95, {"user_id": user.user_id, "user": user})
            if intent == "checkBalance":
                balance = system.get_balance(kwargs["user_id"])
                return AgentResult("PaymentAgent", {"user_id": kwargs["user_id"], "balance": balance}, 0.95, {"user_id": kwargs["user_id"]})
            if intent == "transferMoney":
                tx = system.transfer_money(kwargs["sender_id"], kwargs["receiver_id"], kwargs["amount"])
                return AgentResult("PaymentAgent", asdict(tx), 0.95, {"transaction": tx, "user_id": kwargs["sender_id"]})
            if intent == "showTransactions":
                txs = system.get_transactions(kwargs["user_id"])
                return AgentResult("PaymentAgent", [asdict(tx) for tx in txs], 0.95, {"user_id": kwargs["user_id"]})
            if intent == "requestPayment":
                req = system.request_payment(kwargs["requester_id"], kwargs["payer_id"], kwargs["amount"])
                return AgentResult("PaymentAgent", asdict(req), 0.95, {"payment_request": req, "user_id": kwargs["requester_id"]})
            if intent == "approvePayment":
                req, tx = system.approve_payment_request(kwargs["request_id"])
                return AgentResult("PaymentAgent", {"request": asdict(req), "transaction": asdict(tx)}, 0.95, {"payment_request": req, "transaction": tx, "user_id": req.payer_id})
            if intent == "rejectPayment":
                req = system.reject_payment_request(kwargs["request_id"])
                return AgentResult("PaymentAgent", asdict(req), 0.95, {"payment_request": req, "user_id": req.payer_id})
            return AgentResult("PaymentAgent", "Unsupported payment intent", 0.4)
        except Exception as exc:
            return AgentResult("PaymentAgent", f"Error: {exc}", 0.35, {"error": str(exc)})


class OrchestratorAgent:
    def __init__(self):
        self.system = PaymentSystem()
        self.memory = ShortTermMemory()
        self.router = RouterAgent()
        self.selector = ToolSelector()
        self.payment_agent = PaymentAgent()
        self.fraud_agent = FraudDetectionAgent()
        self.security_agent = SecurityAgent()
        self.explanation_agent = ExplanationAgent()
        self.critic_agent = CriticAgent()
        self.fallback_agent = FallbackAgent()
        self.policy_agent = PolicyAgent()
        self.reflection_agent = ReflectionAgent()

    def run(self, message: str, **kwargs) -> AgentResult:
        route = self.router.run(message)
        intent = kwargs.pop("intent", route.output)
        tool = self.selector.run(intent)

        if tool.output == "FallbackAgent":
            result = self.fallback_agent.run(message)
        elif tool.output == "ExplanationAgent":
            result = self.explanation_agent.run(self.memory)
        elif tool.output == "SecurityAgent":
            result = self.security_agent.run(self.system)
        elif tool.output == "FraudDetectionAgent":
            result = self.fraud_agent.run(self.system, user_id=kwargs.get("user_id"))
        else:
            if intent == "transferMoney":
                policy_result = self.policy_agent.run(self.system, kwargs["sender_id"], kwargs["amount"])
                if not policy_result.metadata["allowed"]:
                    result = policy_result
                else:
                    result = self.payment_agent.run(self.system, intent, **kwargs)
            else:
                result = self.payment_agent.run(self.system, intent, **kwargs)

            tx = result.metadata.get("transaction") if result.metadata else None
            if tx:
                fraud_result = self.fraud_agent.run(self.system, transaction=tx)
                result.metadata["fraud_review"] = fraud_result.output
                if fraud_result.metadata["suspicious"]:
                    security_result = self.security_agent.run(self.system)
                    result.metadata["security_review"] = security_result.output

        critic = self.critic_agent.run(result)
        if critic.metadata["needs_fix"] and result.metadata and "error" in result.metadata:
            result.metadata["reflection"] = self.reflection_agent.run(result.metadata["error"]).output

        self.memory.update(
            action=intent,
            result=result,
            user_id=(result.metadata or {}).get("user_id"),
            transaction=(result.metadata or {}).get("transaction"),
            payment_request=(result.metadata or {}).get("payment_request"),
        )
        return result


def compact(result: AgentResult):
    return {"agent": result.agent_name, "output": result.output, "confidence": result.confidence, "metadata": result.metadata}


print("System and agents are ready")

## בדיקות חובה והרצת הדגמה

הבדיקות הבאות מכסות יצירת משתמשים, יתרות, העברות תקינות ושגויות, בקשות תשלום, זיהוי חשד, הסבר פעולה אחרונה, ביקורת אבטחה ושמירה/טעינה מ-JSON.

In [ ]:
orch = OrchestratorAgent()

def show(title, result):
    print(f"\n--- {title} ---")
    print(compact(result))

# 1. יצירת שני משתמשים ובדיקת יתרות
alice = orch.run("create user", name="Alice", phone_number="050-1111111", initial_balance=8000)
bob = orch.run("create user", name="Bob", phone_number="050-2222222", initial_balance=500)
show("Create Alice", alice)
show("Create Bob", bob)
show("Alice balance", orch.run("check balance", user_id="U001"))
show("Bob balance", orch.run("check balance", user_id="U002"))

# 2. העברת כסף תקינה
show("Valid transfer", orch.run("transfer money", sender_id="U001", receiver_id="U002", amount=700))

# 3. ניסיון להעביר סכום שלילי
show("Negative transfer", orch.run("transfer money", sender_id="U001", receiver_id="U002", amount=-50))

# 4. ניסיון להעביר כסף ללא יתרה מספקת
show("Insufficient balance", orch.run("transfer money", sender_id="U002", receiver_id="U001", amount=99999))

# 5. ניסיון להעביר כסף למשתמש לא קיים
show("Missing receiver", orch.run("transfer money", sender_id="U001", receiver_id="U999", amount=100))

# 6. ניסיון להעביר כסף לעצמך
show("Self transfer", orch.run("transfer money", sender_id="U001", receiver_id="U001", amount=100))

# 7. יצירת בקשת תשלום ואישורה
request = orch.run("request payment", requester_id="U002", payer_id="U001", amount=200)
show("Create payment request", request)
show("Approve payment request", orch.run("approve payment", request_id="R0001"))

# 8. ניסיון לאשר בקשת תשלום פעמיים
show("Approve twice", orch.run("approve payment", request_id="R0001"))

# 9. זיהוי עסקה חשודה
show("Suspicious transfer", orch.run("transfer money", sender_id="U001", receiver_id="U002", amount=4500))
show("Fraud check", orch.run("fraud check", user_id="U001"))

# 10. הסבר על העסקה האחרונה באמצעות הזיכרון
show("Explain last transaction", orch.run("explain last transaction"))

# בדיקת אבטחה
show("Security review", orch.run("security review"))

# אלמנט מתקדם: שמירה וטעינה מ-JSON
orch.system.save_to_json("payment_state_demo.json")
loaded_system = PaymentSystem()
loaded_system.load_from_json("payment_state_demo.json")
print("\n--- JSON persistence ---")
print({"loaded_users": len(loaded_system.users), "loaded_transactions": len(loaded_system.transactions), "loaded_requests": len(loaded_system.payment_requests)})

assert len(orch.system.users) == 2
assert orch.system.wallets["U001"].balance >= 0
assert orch.system.payment_requests["R0001"].status == "approved"
assert orch.memory.last_transaction is not None
print("\nAll required tests ran successfully")

## תשובות לשאלות מסכמות

**כיצד הסוכנים מחלקים את העבודה?**  
RouterAgent מזהה את הכוונה, ToolSelector בוחר את הסוכן המתאים, OrchestratorAgent מנהל את הזרימה, PaymentAgent מפעיל את מערכת התשלומים, FraudDetectionAgent בודק סיכון, SecurityAgent בודק תקינות ואבטחה, ExplanationAgent מסביר פעולות, CriticAgent בוחן איכות תוצאה ו-FallbackAgent מטפל בבקשות לא ברורות.

**כיצד הזיכרון משפר את המערכת?**  
הזיכרון שומר פעולה אחרונה, משתמש אחרון, עסקה אחרונה, בקשת תשלום אחרונה, תוצאה אחרונה ורשימת פעולות אחרונות. כך ניתן להריץ בקשת המשך כמו Explain the last transaction ולקבל הסבר מבוסס הקשר.

**כיצד נמנעות פעולות לא חוקיות?**  
מערכת התשלומים מיישמת כללים עסקיים: אין יתרה התחלתית שלילית, אין העברה בסכום אפס או שלילי, אין העברה לעצמך, אין העברה ללא יתרה מספקת, אין משתמשים לא קיימים, ואין אישור כפול של בקשת תשלום.

**מהם האלמנטים המתקדמים שנבחרו?**  
PolicyAgent מגביל סכום העברה יחידה וסכום יומי. בנוסף, המערכת כוללת שמירה וטעינה מ-JSON של משתמשים, ארנקים, עסקאות, בקשות תשלום ולוג ביקורת.

**כיצד מתבצעת ביקורת ושיפור?**  
CriticAgent בודק תוצאות חלשות או שגיאות. במקרה של שגיאה, ReflectionAgent מוסיף הצעת תיקון למשתמש. לאחר העברה כספית מופעל FraudDetectionAgent, ובעסקה חשודה מופעלת גם ביקורת אבטחה.